# Three Independent Solutions to the Same Climate Task

Yiwen Li (YL), 20-minute seminar. Python reference, CDO-only processing and NCO-only processing. Python provides the final comparison graphics.

Based on [EG's shared notebook](https://github.com/egalytska/Fresh_eyes/blob/393cfcf876fa47c6c231c3be70dbd9f8552fe3a3/stripes_EG.ipynb) and [Abhnil's workshop](https://github.com/abhnil/Fresh_Eyes_on_CMIP_Data_Workshop). The default data are synthetic teaching data, not CMIP observations or ACCESS-CM2 output.

## 0:00-3:00 Scientific contract

All routes start from the same original Kelvin file, never another route's output. Each computes a 1981-2010 calendar-month climatology, monthly anomalies for 1979-2014, a mean anomaly map, an unweighted 65-90N monthly mean and 36 equal-month annual means.

The input checker only reads metadata/data to enforce a shared contract. Python also generates the synthetic fixture. Neither step supplies CDO or NCO with processed fields.

In [ ]:
from pathlib import Path
import os, subprocess, json
if Path.cwd().name == 'notebooks': os.chdir('..')
from IPython.display import display, Image
DATA = os.environ.get('WORKSHOP_INPUT', 'data/demo/tas_demo.nc')
ROOT = 'outputs/independent/monthly'
def run(*args):
    subprocess.run(list(args), check=True)
run('python', 'scripts/generate_demo_data.py', '--output', DATA)
run('python', 'scripts/check_monthly_input.py', DATA)

## 3:00-6:00 Python reference

Read the complete implementation below. `groupby("time.month")` matches January to January. The Arctic mean is arithmetic, not area weighted.

In [ ]:
print(Path('scripts/run_python_monthly_anomaly.py').read_text())
run('python', 'scripts/run_python_monthly_anomaly.py', DATA, ROOT+'/python')

## 6:00-10:00 CDO-only solution

Every data transformation below is a CDO operator. `ymonmean` creates 12 monthly reference means. `ymonsub` subtracts the matching month. `fldmean,weights=false` explicitly disables area weights to reproduce the arithmetic spatial mean. Bare `fldmean` and `fldavg` use area weights. `yearmean` averages the 12 monthly anomalies equally.

In [ ]:
print(Path('scripts/run_cdo_monthly_anomaly.sh').read_text())
run('bash', 'scripts/run_cdo_monthly_anomaly.sh', DATA, ROOT+'/cdo')

## 10:00-14:00 NCO-only solution

NCO selects dates and variables with `ncks`, converts/subtracts with `ncap2`, calculates calendar-month means with strided `ncwa`, assembles climatology with `ncecat`, and averages 12-record annual groups with `ncra --mro`. Bash only orchestrates calls. No CDO command appears in this route.

The baseline starts in January. Zero-based `m=0` is January; stride 12 selects all Januaries. With a January-starting analysis, `t%12` selects the matching reference month. `-d time,,,12,12` groups 12 consecutive records with a 12-record stride. `--mro` retains one result per group.

In [ ]:
print(Path('scripts/run_nco_monthly_anomaly.sh').read_text())
run('bash', 'scripts/run_nco_monthly_anomaly.sh', DATA, ROOT+'/nco')

## 14:00-17:00 Numerical agreement

Python validates CDO against Python and NCO against Python separately. Checks use exact spatial and calendar-coordinate alignment, finite-value masks, shape, the zero monthly baseline, 432 months and 36 years. Absolute tolerance is 0.0003 degC with zero relative tolerance. This allows floating-point differences, not different scientific definitions.

In [ ]:
run('python', 'scripts/compare_three_workflows.py', '--root', ROOT)
report = json.loads(Path(ROOT, 'three_way_validation.json').read_text())
display(report)
assert len(report) == 8 and all(v['passed'] for v in report.values())

## 17:00-19:00 Common Python visualization

Columns are independent Python, CDO and NCO results. All map panels share fixed colour limits, and all stripe panels share one symmetric scale. The map uses latitude/longitude axes rather than Robinson projection to avoid extra map-data downloads in Binder.

In [ ]:
display(Image(filename=str(Path(ROOT, 'three_way_comparison.png')), width=1100))

## 19:00-20:00 Discussion

Which implementation communicates each operation most clearly? NCO needs more explicit calendar bookkeeping for this task; CDO has purpose-built monthly operators. This is not a universal speed or code-length ranking. Count the complete scripts, including setup, rather than comparing a wrapper command with a full Python implementation.

Use CDO **or** NCO to prepare the data. Use Python to inspect, validate and visualize the results.

The older combined script and NCL examples remain historical optional material and are not part of this seminar route. Download your work before closing Binder.